In [73]:
import numpy as np
import pandas as pd
from tensorflow.keras import Sequential
from tensorflow import keras
from tensorflow.keras.models import model_from_json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from utils import evaluate_model

In [75]:
# Features
FEATURE_COLUMNS = [
    "forks_count",
    "subscribers_count",
    "open_issues_count",
    "size",
    "network_count",
    "has_wiki",
    "has_pages",
    "has_issues",
    "topics_count",
    "age_days",
    "language_encoded"
]
TARGET = "stargazers_count"


In [76]:
# load the dataset
df = pd.read_csv("github-repository-data.csv")
df.head()
df["language_encoded"] = pd.factorize(df["language"])[0]
pd.factorize(df["language"])[0]
# Log-transform target
df["log_stars"] = np.log1p(df["stargazers_count"])

X = df[FEATURE_COLUMNS].values
y = df["log_stars"].values

print(X.shape, y.shape)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaled the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

(2000, 11) (2000,)


In [77]:
# model
output_bias = keras.initializers.Constant(y_train.mean())

model = Sequential([
    keras.layers.Input(shape=(len(FEATURE_COLUMNS),)),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1, bias_initializer=output_bias)  # Output layer for regression
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

model.summary()


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_24 (Dense)                │ (None, 64)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,881 (11.25 KB)

 Trainable params: 2,881 (11.25 KB)

 Non-trainable params: 0 (0.00 B)

In [78]:
# train the model
history = model.fit(
    X_train_scaled, 
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1,     # 10% of train for val curve
    callbacks=[
        keras.callbacks.EarlyStopping(
            patience=10, restore_best_weights=True
        )
    ]
)


Epoch 1/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 2.4528 - mae: 1.3525 - val_loss: 2.0844 - val_mae: 1.2591
Epoch 2/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.8603 - mae: 1.1468 - val_loss: 1.5793 - val_mae: 1.0792
Epoch 3/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.5451 - mae: 1.0204 - val_loss: 1.2710 - val_mae: 0.9437
Epoch 4/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.3396 - mae: 0.9374 - val_loss: 1.0877 - val_mae: 0.8665
Epoch 5/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.1348 - mae: 0.8463 - val_loss: 0.9348 - val_mae: 0.7783
Epoch 6/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.0883 - mae: 0.8130 - val_loss: 0.8684 - val_mae: 0.7477
Epoch 7/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.0312 - mae: 0.7931 - val_loss: 0.8036 - val_mae: 0.7038
Epoch 8/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.0278 - mae: 0.7812 - val_loss: 0.7643 - val_mae: 0.6935
Epoch 9/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.986

In [79]:
# evaluate the model
evaluate_model(model, X_test_scaled, y_test, model_name="Neural Network")


13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 

  Neural Network
  R²               : 0.8489
  RMSLE            : 0.6964
  MAE (stars)      :       10,449


{'model': 'Neural Network',
 'r2': 0.8488982683023365,
 'rmsle': 0.6963632517711585,
 'mae': 10448.530979766841}